In [9]:
import os
import re
import glob
from io import BytesIO
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================================================
# BASIC UTILITIES
# ============================================================

def format_filename(text):
    text = str(text).strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_")


def get_excel_basename(source):
    parsed = urlparse(str(source))
    if parsed.scheme in ("http", "https"):
        return os.path.basename(parsed.path)
    return os.path.basename(str(source))


def clean_path(path):
    return path.strip().strip('"').strip("'")


def is_blank(x):
    return pd.isna(x) or str(x).strip() == "" or str(x).strip().lower() in {"nan", "none"}


def is_number_like(x):
    if pd.isna(x):
        return False

    if isinstance(x, (int, float)) and not pd.isna(x):
        return True

    s = str(x).strip()
    if not s:
        return False

    s = (
        s.replace(",", "")
        .replace("€", "")
        .replace("£", "")
        .replace("%", "")
        .replace("(", "-")
        .replace(")", "")
    )

    try:
        float(s)
        return True
    except Exception:
        return False


def clean_numeric_series(series):
    s = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("€", "", regex=False)
        .str.replace("£", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("(", "-", regex=False)
        .str.replace(")", "", regex=False)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA})
    )
    return pd.to_numeric(s, errors="coerce")


def make_unique(cols):
    seen = {}
    out = []

    for c in cols:
        c = str(c).strip()

        if c in seen:
            seen[c] += 1
            out.append(f"{c}_{seen[c]}")
        else:
            seen[c] = 0
            out.append(c)

    return out


def trim_empty_edges(df):
    return df.dropna(how="all").dropna(axis=1, how="all")


def normalise_key(text):
    return re.sub(r"[^a-z0-9]+", "", str(text).strip().lower())


def strip_duplicate_suffix(text):
    """
    Converts:
    Year_1 -> Year
    Value_2 -> Value
    """
    return re.sub(r"_(\d+)$", "", str(text).strip())


def base_header_key(text):
    """
    Converts:
    Year_1 -> year
    AccidentQuarter_1 -> accidentquarter
    Value_2 -> value
    """
    return normalise_key(strip_duplicate_suffix(text))


def extract_year(text):
    m = re.search(r"\b((?:19|20)\d{2})\b", str(text))
    return m.group(1) if m else pd.NA


def extract_period_type(text):
    s = str(text).strip()

    m = re.search(r"\b(H[1-2])\b", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    m = re.search(r"\b(Q[1-4])\b", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    return pd.NA


def looks_like_footer(value):
    if pd.isna(value):
        return False

    s = str(value).strip().lower()

    footer_terms = [
        "coverage",
        "source",
        "note",
        "notes",
        "total coverage",
        "data source",
        "prepared by",
        "methodology",
        "definitions",
        "disclaimer",
    ]

    return any(term in s for term in footer_terms)


def looks_like_footer_row(row_values):
    joined = " ".join([str(v).strip().lower() for v in row_values if not is_blank(v)])

    footer_terms = [
        "coverage",
        "source",
        "note",
        "notes",
        "total coverage",
        "data source",
        "prepared by",
        "methodology",
        "definitions",
        "disclaimer",
    ]

    return any(term in joined for term in footer_terms)


def looks_like_date_or_period(value):
    """
    Handles:
    - 2024
    - H1 2024
    - H2 2024
    - 2024 H1
    - Q1 2024
    - 2024 Q1
    - January 2024
    - normal dates
    """
    if pd.isna(value):
        return False

    s = str(value).strip()

    if not s:
        return False

    if re.fullmatch(r"\d{4}", s):
        return True

    if re.fullmatch(r"H[1-2]\s*\d{4}", s, flags=re.IGNORECASE):
        return True

    if re.fullmatch(r"\d{4}\s*H[1-2]", s, flags=re.IGNORECASE):
        return True

    if re.fullmatch(r"Q[1-4]\s*\d{4}", s, flags=re.IGNORECASE):
        return True

    if re.fullmatch(r"\d{4}\s*Q[1-4]", s, flags=re.IGNORECASE):
        return True

    if re.fullmatch(r"[A-Za-z]{3,9}\s+\d{4}", s):
        return True

    try:
        pd.to_datetime(s, errors="raise")
        return True
    except Exception:
        return False


def looks_like_accident_quarter(value):
    """
    Handles:
    - 201003
    - 201006
    - 201009
    - 201012
    """
    if pd.isna(value):
        return False

    s = str(value).strip()
    s = re.sub(r"\.0$", "", s)

    return bool(re.fullmatch(r"(19|20)\d{2}(03|06|09|12)", s))


def is_year_col(col):
    if pd.isna(col):
        return False

    if isinstance(col, (int, float)):
        try:
            y = int(col)
            return 1900 <= y <= 2100
        except Exception:
            return False

    s = str(col).strip()

    if re.fullmatch(r"\d{4}", s):
        return True

    return bool(re.search(r"\b(19|20)\d{2}\b", s))


# ============================================================
# STANDARDISE COLUMNS
# ============================================================

def standardise_column_names(df):
    df = df.copy()
    rename_map = {}

    for c in df.columns:
        raw = strip_duplicate_suffix(c)
        key = normalise_key(raw)

        if key == "year":
            rename_map[c] = "year"
        elif key in {
            "settledyear",
            "date",
            "period",
            "reportingperiod",
            "halfyear",
            "calendarperiod",
            "financialperiod",
        }:
            rename_map[c] = "date"
        elif key in {"accidentquarter", "accidentqtr"}:
            rename_map[c] = "AccidentQuarter"
        elif key == "originyear":
            rename_map[c] = "OriginYear"
        elif key == "underwritingyear":
            rename_map[c] = "UnderwritingYear"
        elif key == "claimtype":
            rename_map[c] = "ClaimType"
        elif key == "covertype":
            rename_map[c] = "CoverType"
        elif key in {"measure", "variable", "metric"}:
            rename_map[c] = "measure"
        elif key in {"measureid", "histexpmeasureid"}:
            rename_map[c] = "measure_id"
        elif key in {"value", "values", "amount"}:
            rename_map[c] = "value"
        else:
            rename_map[c] = str(raw).strip()

    df = df.rename(columns=rename_map)
    df.columns = make_unique(df.columns)

    return df


def ensure_required_time_and_value(df):
    df = df.copy()
    df = standardise_column_names(df)

    if df.empty:
        return pd.DataFrame()

    if "value" not in df.columns:
        return pd.DataFrame()

    df["value"] = clean_numeric_series(df["value"])
    df = df.dropna(subset=["value"]).reset_index(drop=True)

    if df.empty:
        return pd.DataFrame()

    time_cols = [
        c for c in df.columns
        if normalise_key(c) in {
            "year",
            "date",
            "accidentquarter",
            "originyear",
            "underwritingyear",
        }
    ]

    if not time_cols:
        return pd.DataFrame()

    if "date" in df.columns:
        if "year" not in df.columns:
            years = df["date"].astype(str).str.extract(r"((?:19|20)\d{2})", expand=False)
            if years.notna().mean() >= 0.6:
                df["year"] = years

        if "period_type" not in df.columns:
            df["period_type"] = df["date"].apply(extract_period_type)

    if "AccidentQuarter" in df.columns:
        df["AccidentQuarter"] = (
            df["AccidentQuarter"]
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)
        )

        if "year" not in df.columns:
            years = df["AccidentQuarter"].astype(str).str.extract(r"^((?:19|20)\d{2})", expand=False)
            if years.notna().mean() >= 0.6:
                df["year"] = years

    if "year" in df.columns:
        df["year"] = df["year"].astype(str).str.extract(r"((?:19|20)\d{2})", expand=False)

    return df


def drop_metadata_columns(df):
    df = df.copy()
    df = ensure_required_time_and_value(df)

    if df.empty:
        return pd.DataFrame()

    metadata_cols = {
        "source_file",
        "sheet_name",
        "table_index",
        "table_title",
        "processing_method",
        "output_file",
        "method_used",
        "status",
        "source_block",
        "source_group",
    }

    metadata_keys = {normalise_key(x) for x in metadata_cols}

    keep_cols = [
        c for c in df.columns
        if normalise_key(c) not in metadata_keys
    ]

    return df[keep_cols] if keep_cols else pd.DataFrame()


# ============================================================
# SPLITTING HELPERS
# ============================================================

def split_by_blank_cols(df, blank_run=1, min_cols=2):
    df = df.copy()
    blank_cols = []

    for c in range(df.shape[1]):
        blank_cols.append(all(is_blank(v) for v in df.iloc[:, c].values))

    blocks = []
    start = None
    run = 0

    for j, is_b in enumerate(blank_cols):
        if not is_b:
            if start is None:
                start = j
            run = 0
        else:
            if start is not None:
                run += 1

                if run >= blank_run:
                    end = j - run + 1
                    block = trim_empty_edges(df.iloc[:, start:end].copy()).reset_index(drop=True)

                    if not block.empty and block.shape[1] >= min_cols:
                        blocks.append((block, start, end))

                    start = None
                    run = 0

    if start is not None:
        end = df.shape[1]
        block = trim_empty_edges(df.iloc[:, start:end].copy()).reset_index(drop=True)

        if not block.empty and block.shape[1] >= min_cols:
            blocks.append((block, start, end))

    return blocks


def split_by_blank_rows(df, blank_run=1, min_rows=3):
    df = df.copy()
    blank_rows = df.apply(lambda r: all(is_blank(v) for v in r.values), axis=1).tolist()

    blocks = []
    start = None
    run = 0

    for i, is_b in enumerate(blank_rows):
        if not is_b:
            if start is None:
                start = i
            run = 0
        else:
            if start is not None:
                run += 1

                if run >= blank_run:
                    end = i - run + 1
                    block = trim_empty_edges(df.iloc[start:end].copy()).reset_index(drop=True)

                    if not block.empty and len(block) >= min_rows and block.shape[1] >= 2:
                        blocks.append((block, start, end))

                    start = None
                    run = 0

    if start is not None:
        end = len(df)
        block = trim_empty_edges(df.iloc[start:end].copy()).reset_index(drop=True)

        if not block.empty and len(block) >= min_rows and block.shape[1] >= 2:
            blocks.append((block, start, end))

    return blocks


def get_initial_side_blocks(sheet_df):
    """
    First split using blank columns.
    If this does not split the sheet, repeated-header splitting later catches UltData.
    """
    df = trim_empty_edges(sheet_df.copy()).reset_index(drop=True)

    if df.empty:
        return []

    col_blocks = split_by_blank_cols(df, blank_run=1, min_cols=2)

    if col_blocks and len(col_blocks) > 1:
        return col_blocks

    return [(df, 0, df.shape[1])]


# ============================================================
# HEADER DETECTION
# ============================================================

def row_nonblank_values(row):
    return [v for v in row if not is_blank(v)]


def row_text_ratio(row):
    vals = row_nonblank_values(row)

    if not vals:
        return 0

    return sum(not is_number_like(v) for v in vals) / len(vals)


def row_numeric_ratio(row):
    vals = row_nonblank_values(row)

    if not vals:
        return 0

    return sum(is_number_like(v) for v in vals) / len(vals)


def row_known_header_hits(row):
    vals = [str(v).strip() for v in row_nonblank_values(row)]

    known = {
        "year",
        "settled year",
        "accidentquarter",
        "accident quarter",
        "measure",
        "claim type",
        "cover type",
        "value",
        "values",
        "amount",
        "date",
        "period",
    }

    known_keys = {normalise_key(x) for x in known}

    return sum(normalise_key(v) in known_keys for v in vals)


def find_header_row(df, max_scan=25):
    best_idx = 0
    best_score = -999

    for i in range(min(max_scan, len(df))):
        row = df.iloc[i].tolist()
        vals = row_nonblank_values(row)

        if len(vals) < 2:
            continue

        known_hits = row_known_header_hits(row)
        text_ratio = row_text_ratio(row)
        numeric_ratio = row_numeric_ratio(row)

        score = 0
        score += known_hits * 15
        score += len(vals) * 1.5
        score += text_ratio * 4

        if numeric_ratio >= 0.6 and known_hits == 0:
            score -= 12

        if len(vals) <= 2 and text_ratio == 1 and known_hits == 0:
            score -= 8

        if score > best_score:
            best_score = score
            best_idx = i

    return best_idx


def promote_header(raw_block):
    df = trim_empty_edges(raw_block.copy()).reset_index(drop=True)

    if df.empty:
        return pd.DataFrame()

    header_idx = find_header_row(df)
    header = df.iloc[header_idx].tolist()

    cols = []

    for j, v in enumerate(header):
        if is_blank(v):
            cols.append(f"column_{j}")
        else:
            cols.append(str(v).strip())

    cols = make_unique(cols)

    data = df.iloc[header_idx + 1:].copy().reset_index(drop=True)
    data = trim_empty_edges(data).reset_index(drop=True)

    if data.empty:
        return pd.DataFrame()

    data.columns = cols[:data.shape[1]]
    data = data.dropna(how="all").reset_index(drop=True)

    return data


# ============================================================
# REPEATED HEADER GROUP SPLITTING
# ============================================================

def split_promoted_df_by_repeated_groups(df):
    """
    Critical UltData fix.

    Splits:
    year | AccidentQuarter | measure | ClaimType | value | Year_1 | AccidentQuarter_1 | Measure_1 | CoverType | Value_1

    into:
    1. year | AccidentQuarter | measure | ClaimType | value
    2. Year_1 | AccidentQuarter_1 | Measure_1 | CoverType | Value_1
    """
    df = df.copy()

    if df.empty or df.shape[1] < 4:
        return [df]

    cols = list(df.columns)
    anchor_positions = []

    for i, c in enumerate(cols):
        key = base_header_key(c)

        if key in {"year", "date", "settledyear"}:
            anchor_positions.append(i)

    if len(anchor_positions) <= 1:
        return [df]

    split_dfs = []

    for idx, start_col in enumerate(anchor_positions):
        end_col = anchor_positions[idx + 1] if idx + 1 < len(anchor_positions) else len(cols)

        part = df.iloc[:, start_col:end_col].copy()
        part = part.dropna(axis=1, how="all")
        part = part.dropna(how="all").reset_index(drop=True)

        if part.empty or part.shape[1] < 2:
            continue

        part = standardise_column_names(part)

        has_value = "value" in part.columns
        has_time = any(
            normalise_key(c) in {"year", "date", "accidentquarter", "originyear", "underwritingyear"}
            for c in part.columns
        )

        if has_value and has_time:
            split_dfs.append(part)

    return split_dfs if split_dfs else [df]


# ============================================================
# CLEANING AND TIDYING
# ============================================================

def remove_footer_rows(df):
    df = df.copy().reset_index(drop=True)

    for i in range(len(df)):
        if looks_like_footer_row(df.iloc[i].values):
            return df.iloc[:i].copy().reset_index(drop=True)

    return df


def remove_repeated_header_rows(df):
    df = df.copy().reset_index(drop=True)
    col_keys = {normalise_key(c) for c in df.columns}

    drop_idx = []

    for i in range(len(df)):
        vals = row_nonblank_values(df.iloc[i].tolist())

        if not vals:
            continue

        hits = sum(normalise_key(v) in col_keys for v in vals)

        if hits >= 2:
            drop_idx.append(i)

    if drop_idx:
        df = df.drop(index=drop_idx).reset_index(drop=True)

    return df


def tidy_already_long(df):
    """
    Handles:
    Year | AccidentQuarter | Measure | ClaimType | Value
    Year | AccidentQuarter | Measure | CoverType | Value
    """
    df = df.copy()
    df = standardise_column_names(df)
    df = remove_repeated_header_rows(df)

    if "value" not in df.columns:
        return pd.DataFrame()

    has_time = any(
        normalise_key(c) in {"year", "date", "accidentquarter", "originyear", "underwritingyear"}
        for c in df.columns
    )

    if not has_time:
        return pd.DataFrame()

    return drop_metadata_columns(df)


def tidy_first_col_period_table(df):
    """
    Handles:
    Settled Year | Number of Claimants Settled | Total Settled Cost (€)

    Output:
    date | measure | value | year | period_type
    """
    df = df.copy()
    df = standardise_column_names(df)

    if df.empty or df.shape[1] < 2:
        return pd.DataFrame()

    first_col = df.columns[0]
    sample = df[first_col].dropna().head(50)

    if len(sample) == 0:
        return pd.DataFrame()

    time_ratio = sample.apply(looks_like_date_or_period).mean()

    if time_ratio < 0.5:
        return pd.DataFrame()

    if first_col != "date":
        df = df.rename(columns={first_col: "date"})

    df = df[df["date"].apply(looks_like_date_or_period)].reset_index(drop=True)

    numeric_cols = []

    for col in df.columns:
        if col == "date":
            continue

        sample = df[col].dropna().head(50)

        if len(sample) == 0:
            continue

        ratio = sum(is_number_like(v) for v in sample) / len(sample)

        if ratio >= 0.5:
            numeric_cols.append(col)

    if not numeric_cols:
        return pd.DataFrame()

    for col in numeric_cols:
        df[col] = clean_numeric_series(df[col])

    long_df = df.melt(
        id_vars=["date"],
        value_vars=numeric_cols,
        var_name="measure",
        value_name="value"
    )

    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)
    long_df["year"] = long_df["date"].apply(extract_year)
    long_df["period_type"] = long_df["date"].apply(extract_period_type)

    return drop_metadata_columns(long_df)


def tidy_wide_year_table(df):
    """
    Handles:
    Measure | 2019 | 2020 | 2021
    """
    df = df.copy()
    df = standardise_column_names(df)

    year_cols = [c for c in df.columns if is_year_col(c)]

    if len(year_cols) < 2:
        return pd.DataFrame()

    non_year_cols = [c for c in df.columns if c not in year_cols]

    if not non_year_cols:
        return pd.DataFrame()

    rename_map = {non_year_cols[0]: "measure"}

    for idx, c in enumerate(non_year_cols[1:], start=2):
        rename_map[c] = f"dimension_{idx}"

    df = df.rename(columns=rename_map)

    for yc in year_cols:
        df[yc] = clean_numeric_series(df[yc])

    id_vars = [c for c in df.columns if c not in year_cols]

    long_df = df.melt(
        id_vars=id_vars,
        value_vars=year_cols,
        var_name="year",
        value_name="value"
    )

    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)
    long_df["year"] = long_df["year"].apply(extract_year)

    return drop_metadata_columns(long_df)


def tidy_generic_numeric_table(df):
    """
    Last fallback:
    Keeps text columns as dimensions and melts numeric columns.
    """
    df = df.copy()
    df = standardise_column_names(df)

    numeric_cols = []

    for col in df.columns:
        if normalise_key(col) in {"year", "date", "accidentquarter", "originyear", "underwritingyear"}:
            continue

        sample = df[col].dropna().head(50)

        if len(sample) == 0:
            continue

        ratio = sum(is_number_like(v) for v in sample) / len(sample)

        if ratio >= 0.6:
            numeric_cols.append(col)

    if not numeric_cols:
        return pd.DataFrame()

    for col in numeric_cols:
        df[col] = clean_numeric_series(df[col])

    id_vars = [c for c in df.columns if c not in numeric_cols]

    long_df = df.melt(
        id_vars=id_vars,
        value_vars=numeric_cols,
        var_name="measure",
        value_name="value"
    )

    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)

    return drop_metadata_columns(long_df)


def tidy_one_promoted_table(df):
    df = df.copy()
    df = remove_footer_rows(df)
    df = trim_empty_edges(df).reset_index(drop=True)

    if df.empty:
        return pd.DataFrame()

    # 1. Already-long layout, e.g. UltData
    out = tidy_already_long(df)
    if out is not None and not out.empty:
        return out

    # 2. First-column period layout, e.g. H1/H2 tables
    out = tidy_first_col_period_table(df)
    if out is not None and not out.empty:
        return out

    # 3. Wide year layout
    out = tidy_wide_year_table(df)
    if out is not None and not out.empty:
        return out

    # 4. Generic fallback
    out = tidy_generic_numeric_table(df)
    if out is not None and not out.empty:
        return out

    return pd.DataFrame()


def process_sheet_into_tables(sheet_df, selected_source, sheet_name):
    outputs = []
    table_idx = 0

    initial_side_blocks = get_initial_side_blocks(sheet_df)

    if len(initial_side_blocks) > 1:
        print(f"  Detected {len(initial_side_blocks)} blank-column side-by-side blocks")

    for side_block_num, side_info in enumerate(initial_side_blocks, start=1):
        side_df, col_start, col_end = side_info
        side_df = trim_empty_edges(side_df).reset_index(drop=True)

        if side_df.empty:
            continue

        row_blocks = split_by_blank_rows(side_df, blank_run=1, min_rows=3)

        if not row_blocks:
            row_blocks = [(side_df, 0, len(side_df))]

        for row_block_num, row_info in enumerate(row_blocks, start=1):
            raw_block, row_start, row_end = row_info
            raw_block = trim_empty_edges(raw_block).reset_index(drop=True)

            if raw_block.empty:
                continue

            promoted = promote_header(raw_block)

            if promoted.empty:
                continue

            # Critical: split adjacent repeated groups AFTER header promotion
            promoted_groups = split_promoted_df_by_repeated_groups(promoted)

            if len(promoted_groups) > 1:
                print(
                    f"  Detected {len(promoted_groups)} repeated-header groups "
                    f"in block {side_block_num}.{row_block_num}"
                )

            for group_num, group_df in enumerate(promoted_groups, start=1):
                tidy_df = tidy_one_promoted_table(group_df)

                if tidy_df is None or tidy_df.empty:
                    continue

                table_idx += 1

                tidy_df.insert(0, "source_file", get_excel_basename(selected_source))
                tidy_df.insert(1, "sheet_name", sheet_name)
                tidy_df.insert(2, "table_index", table_idx)
                tidy_df.insert(3, "source_block", side_block_num)
                tidy_df.insert(4, "source_group", group_num)

                tidy_df = drop_metadata_columns(tidy_df)

                if tidy_df is None or tidy_df.empty:
                    continue

                outputs.append({
                    "df": tidy_df,
                    "table_index": table_idx,
                    "source_block": side_block_num,
                    "source_group": group_num,
                    "method": "tidy_engine"
                })

    return outputs


# ============================================================
# INPUT HELPERS
# ============================================================

def choose_input_mode():
    print("Choose input method:")
    print("1. Scrape Excel files from a webpage")
    print("2. Use a single local Excel file")
    print("3. Use a folder of local Excel files")

    while True:
        choice = input("\nEnter 1, 2, or 3: ").strip()

        if choice in {"1", "2", "3"}:
            return choice

        print("Invalid choice.")


def choose_from_numbered_list(items, prompt):
    while True:
        try:
            choice = int(input(prompt).strip())

            if 1 <= choice <= len(items):
                return items[choice - 1]

            print("Please enter a valid number from the list.")
        except Exception:
            print("Please enter a valid number.")


def choose_sheets(sheet_names):
    print("\nSheets available:")
    print("-" * 40)

    for i, name in enumerate(sheet_names, start=1):
        print(f"{i}. {name}")

    print("-" * 40)
    print(f"Total sheets: {len(sheet_names)}")

    with open("sheet_list.txt", "w", encoding="utf-8") as f:
        for i, name in enumerate(sheet_names, start=1):
            f.write(f"{i}. {name}\n")

    print("Full sheet list saved to: sheet_list.txt")

    print("\nOptions:")
    print(" - Enter a single number, e.g. 2")
    print(" - Enter multiple numbers separated by commas, e.g. 1,3,4")
    print(" - Enter 'all' to process all sheets")

    while True:
        selection = input("\nSelect sheets: ").strip().lower()

        if selection == "all":
            return sheet_names

        try:
            indices = [int(x.strip()) for x in selection.split(",")]

            if all(1 <= i <= len(sheet_names) for i in indices):
                chosen = []

                for i in indices:
                    nm = sheet_names[i - 1]

                    if nm not in chosen:
                        chosen.append(nm)

                return chosen

            print("Invalid selection.")
        except Exception:
            print("Please enter valid numbers or 'all'.")


# ============================================================
# MAIN
# ============================================================

def main():
    headers = {"User-Agent": "Mozilla/5.0"}
    mode = choose_input_mode()

    selected_source = None
    df_dict = None

    # --------------------------------------------------------
    # MODE 1: Web scrape
    # --------------------------------------------------------
    if mode == "1":
        page_url = input("\nInsert webpage link: ").strip()

        response = requests.get(page_url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        excel_links = []

        for link in soup.find_all("a"):
            href = link.get("href")

            if href and any(ext in href.lower() for ext in [".xlsx", ".xlsm"]):
                excel_links.append(urljoin(page_url, href))

        excel_links = list(dict.fromkeys(excel_links))

        if not excel_links:
            print("No Excel files found on the webpage.")
            return

        print("\nExcel files found:\n")

        for i, link in enumerate(excel_links, start=1):
            print(f"{i}. {get_excel_basename(link)}")

        selected_source = choose_from_numbered_list(excel_links, "\nSelect file number: ")

        file_data = requests.get(selected_source, headers=headers)
        file_data.raise_for_status()

        df_dict = pd.read_excel(
            BytesIO(file_data.content),
            sheet_name=None,
            engine="openpyxl",
            header=None,
            dtype=object
        )

    # --------------------------------------------------------
    # MODE 2: Single local file
    # --------------------------------------------------------
    elif mode == "2":
        selected_source = clean_path(input("\nEnter full file path: "))

        if not os.path.exists(selected_source):
            print("File not found. Check the path you pasted.")
            return

        df_dict = pd.read_excel(
            selected_source,
            sheet_name=None,
            engine="openpyxl",
            header=None,
            dtype=object
        )

    # --------------------------------------------------------
    # MODE 3: Folder of local files
    # --------------------------------------------------------
    elif mode == "3":
        folder_path = clean_path(input("\nEnter folder path: "))

        if not os.path.isdir(folder_path):
            print("That folder path does not exist.")
            return

        files = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith((".xlsx", ".xlsm"))
        ]

        files.sort()

        if not files:
            print("No Excel files (.xlsx/.xlsm) found in that folder.")
            return

        print("\nExcel files available:\n")

        for i, f in enumerate(files, start=1):
            print(f"{i}. {f}")

        chosen_file = choose_from_numbered_list(files, "\nSelect file number: ")
        selected_source = os.path.join(folder_path, chosen_file)

        df_dict = pd.read_excel(
            selected_source,
            sheet_name=None,
            engine="openpyxl",
            header=None,
            dtype=object
        )

    else:
        print("Invalid mode.")
        return

    sheet_names = list(df_dict.keys())
    selected_sheets = choose_sheets(sheet_names)

    print("\nSelected sheets:", selected_sheets)

    os.makedirs("tidy_outputs", exist_ok=True)

    base_name = os.path.splitext(get_excel_basename(selected_source))[0]
    file_part = format_filename(base_name)

    log_rows = []

    for sheet_name in selected_sheets:
        print("\nProcessing sheet:", sheet_name)

        sheet_df = df_dict[sheet_name]
        sheet_part = format_filename(sheet_name)

        # Remove old outputs for this sheet so stale single CSVs do not confuse results
        old_pattern = os.path.join("tidy_outputs", f"{file_part}__{sheet_part}*.csv")

        for old_file in glob.glob(old_pattern):
            try:
                os.remove(old_file)
            except Exception:
                pass

        try:
            table_outputs = process_sheet_into_tables(
                sheet_df=sheet_df,
                selected_source=selected_source,
                sheet_name=sheet_name
            )

            if not table_outputs:
                print("  Skipped: no valid tidy tables found")

                log_rows.append({
                    "source_file": get_excel_basename(selected_source),
                    "sheet_name": sheet_name,
                    "table_index": "",
                    "source_block": "",
                    "source_group": "",
                    "output_file": "",
                    "method_used": "none",
                    "status": "skipped"
                })

                continue

            # If more than one table is detected, always save table_1, table_2, etc.
            # This is what UltData needs.
            if len(table_outputs) == 1:
                output_name = f"{file_part}__{sheet_part}.csv"
                output_path = os.path.join("tidy_outputs", output_name)

                final_df = table_outputs[0]["df"]
                final_df.to_csv(output_path, index=False)

                print(f"  Saved: {output_name}")
                print("  Output columns:", list(final_df.columns))

                log_rows.append({
                    "source_file": get_excel_basename(selected_source),
                    "sheet_name": sheet_name,
                    "table_index": table_outputs[0]["table_index"],
                    "source_block": table_outputs[0]["source_block"],
                    "source_group": table_outputs[0]["source_group"],
                    "output_file": output_name,
                    "method_used": table_outputs[0]["method"],
                    "status": "saved"
                })

            else:
                for out_idx, item in enumerate(table_outputs, start=1):
                    output_name = f"{file_part}__{sheet_part}__table_{out_idx}.csv"
                    output_path = os.path.join("tidy_outputs", output_name)

                    final_df = item["df"]
                    final_df.to_csv(output_path, index=False)

                    print(f"  Table {out_idx} saved -> {output_name}")
                    print("  Output columns:", list(final_df.columns))

                    log_rows.append({
                        "source_file": get_excel_basename(selected_source),
                        "sheet_name": sheet_name,
                        "table_index": item["table_index"],
                        "source_block": item["source_block"],
                        "source_group": item["source_group"],
                        "output_file": output_name,
                        "method_used": item["method"],
                        "status": "saved"
                    })

        except Exception as e:
            print(f"  Error processing sheet {sheet_name}: {e}")

            log_rows.append({
                "source_file": get_excel_basename(selected_source),
                "sheet_name": sheet_name,
                "table_index": "",
                "source_block": "",
                "source_group": "",
                "output_file": "",
                "method_used": "error",
                "status": str(e)
            })

    if log_rows:
        log_df = pd.DataFrame(log_rows)
        log_path = os.path.join("tidy_outputs", "processing_log.csv")
        log_df.to_csv(log_path, index=False)

        print("\nSaved log: tidy_outputs/processing_log.csv")

    print("\nFinished processing.")
    print("Final outputs saved in: tidy_outputs/")


if __name__ == "__main__":
    main()


Choose input method:
1. Scrape Excel files from a webpage
2. Use a single local Excel file
3. Use a folder of local Excel files

Excel files found:

1. data-annex-ncid-private-motor-insurance-mid-year-2025-settled.xlsx
2. data-annex-ncid-private-motor-insurance-mid-year-2025.xlsx
3. annex-private-motor-insurance-report-7.xlsx
4. data-annex-ncid-private-motor-insurance-mid-year-2024-settled-claims.xlsx
5. data-annex-ncid-private-motor-insurance-mid-year-2024.xlsx
6. annex-ncid-private-motor-insurance-report-6.xlsx
7. data-annex-ncid-private-motor-insurance-mid-year-report-2.xlsx
8. annex-ncid-private-motor-insurance-report-5.xlsx
9. data-annex-ncid-private-motor-insurance-mid-year-report-1.xlsx
10. annex-private-motor-insurance-report-4.xlsx
11. annex-private-motor-insurance-report-3.xlsx
12. annex-private-motor-insurance-report-2-data.xlsx
13. annex-2-private-motor-insurance-report-2.xlsx
14. annex-1---settlement-channels-2015-to-2018.xlsx
15. annex-2---private-motor-insurance-report-1